In [0]:
%run /Workspace/Users/gustavosousa.md20@gmail.com/3752-RETREINO/00_struct_table

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


<function __main__.dist_haversine(lat_o, lon_o, lat_d, lon_d)>

In [0]:
df = spark.table("tbl_ml").toPandas()

# Não recomendado
# df.drop(columns=["periodo"], inplace=True)

In [0]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["vlr_pago"])
Y = df["vlr_pago"]

X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=42
)

In [0]:
colunas_minmax = [
    'dia',
    'dia_semana',
    'mes',
    'hora',
    'distancia',
    'latitude_origem',
    'longitude_origem',
    'latitude_destino',
    'longitude_destino'
]

In [0]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler

# Desabilitar o MLFLOW automatico do Databricks
import mlflow
mlflow.autolog(disable=True)

# Definir os processadores
preprocessador = ColumnTransformer(
    transformers=[
        ('drop', 'drop', ['periodo']),
        ('minmax', MinMaxScaler(), colunas_minmax)
    ],
    remainder='passthrough'  # mantém as outras colunas sem alteração
)

# Combinar os processadores + pipeline
pipeline = Pipeline(
    steps=[
        ('preprocessamento', preprocessador),
    ]
)

# Gerar o objeto do pipeline
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('drop', 'drop', ['periodo']),
                                                 ('minmax', MinMaxScaler(),
                                                  ['dia', 'dia_semana', 'mes',
                                                   'hora', 'distancia',
                                                   'latitude_origem',
                                                   'longitude_origem',
                                                   'latitude_destino',
                                                   'longitude_destino'])]))])

In [0]:
X_train.head(5)

,dia,dia_semana,mes,hora,periodo,distancia,latitude_origem,longitude_origem,latitude_destino,longitude_destino
55,14,2,7,10,1,4.097780,-23.544797,-46.593638,-23.528420,-46.629647
22,28,5,8,17,2,2.805872,-23.541157,-46.607837,-23.520587,-46.591897
76,5,4,11,8,1,4.405999,-23.533713,-46.672355,-23.504735,-46.642881
44,1,6,8,9,1,6.293622,-23.604522,-46.652704,-23.623673,-46.594574
72,10,2,3,15,2,4.496438,-23.562517,-46.591200,-23.578353,-46.550605


In [0]:
pipeline

Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('drop', 'drop', ['periodo']),
                                                 ('minmax', MinMaxScaler(),
                                                  ['dia', 'dia_semana', 'mes',
                                                   'hora', 'distancia',
                                                   'latitude_origem',
                                                   'longitude_origem',
                                                   'latitude_destino',
                                                   'longitude_destino'])]))])

In [0]:
import pandas as pd
pd.DataFrame(pipeline.transform(X_train))

,0,1,2,3,4,5,6,7,8
0,0.433333,0.166667,0.555556,0.200000,0.522261,0.588941,0.931804,0.656521,0.517137
1,0.900000,0.666667,0.666667,0.666667,0.241187,0.624834,0.779545,0.704838,0.747749
2,0.133333,0.500000,1.000000,0.066667,0.589319,0.698248,0.087727,0.802614,0.436285
3,0.000000,0.833333,0.666667,0.133333,1.000000,0.000000,0.298449,0.068988,0.731392
4,0.300000,0.166667,0.111111,0.533333,0.608995,0.414208,0.957947,0.348526,1.000000
...,...,...,...,...,...,...,...,...,...
78,0.333333,0.333333,0.111111,0.800000,0.656468,0.531258,1.000000,0.759630,0.664246
79,0.566667,0.666667,0.777778,0.400000,0.546205,0.764813,0.160777,0.849808,0.452560
80,0.733333,0.666667,0.888889,0.200000,0.524668,0.737908,0.862337,0.461381,0.837845
81,0.700000,0.333333,0.555556,0.600000,0.524707,0.383303,0.899401,0.238416,0.580108


In [0]:
import pandas as pd

pd.DataFrame( pipeline.transform(X_train) ).describe().T

,count,mean,std,min,25%,50%,75%,max
0,83.0,0.419679,0.298417,0.0,0.183333,0.333333,0.700000,1.0
1,83.0,0.558233,0.299541,0.0,0.333333,0.500000,0.833333,1.0
2,83.0,0.672021,0.242877,0.0,0.555556,0.666667,0.888889,1.0
3,83.0,0.410442,0.249983,0.0,0.200000,0.400000,0.600000,1.0
4,83.0,0.473124,0.195070,0.0,0.340494,0.522261,0.587924,1.0
5,83.0,0.535757,0.250556,0.0,0.329923,0.555949,0.750497,1.0
6,83.0,0.540930,0.295233,0.0,0.234244,0.582255,0.807175,1.0
7,83.0,0.510763,0.257211,0.0,0.303392,0.537481,0.725423,1.0
8,83.0,0.507766,0.240781,0.0,0.338597,0.490708,0.725694,1.0
